# JWST artificial redshifting — portable workflow

This self-contained notebook measures how a galaxy's recoverable structure
changes when it is artificially observed at another redshift and injected into
a real JWST background. It implements the scientifically separated chain:

`native/input → native-clean model → target-redshift clean model → target-redshift real-background injection → structural recovery`.

It uses Lenstronomy to render structural truth and Galight to recover it. It
does **not** use a homemade analytic Sersic image renderer.

## Quick start

1. Set `DEMO_MODE = True` in the next settings cell and choose **Run All**.
2. For data, set `DEMO_MODE = False`, then edit only that settings cell.
3. Supply distinct native/source and target/background products (or explicitly
   enable the same-dataset convenience mode).
4. Read the preflight PASS / WARNING / FAIL table before interpreting results.
5. Results, figures, config, provenance, logs, and checkpoints are saved in
   `OUTPUT_DIR`.

The default real-background mode injects a deterministic source into the
already noisy target SCI mosaic. It never draws a second sky/read/correlated
background-noise realization. Injected-source Poisson noise is optional and
only available with physically valid count/exposure metadata.


## 1. Optional dependency check

In [ ]:
# This cell does not modify your environment.
import importlib.util

REQUIRED_PACKAGES = {
    'numpy': 'numpy', 'scipy': 'scipy', 'astropy': 'astropy',
    'matplotlib': 'matplotlib', 'photutils': 'photutils',
    'lenstronomy': 'lenstronomy', 'galight': 'galight',
}
missing_packages = [pip for module, pip in REQUIRED_PACKAGES.items()
                    if importlib.util.find_spec(module) is None]
if missing_packages:
    print('Missing packages:', ', '.join(missing_packages))
    print('Suggested command:')
    print('pip install ' + ' '.join(missing_packages))
    print('Install them in a suitable environment, restart the kernel, and Run All again.')
else:
    print('Environment check: PASS — required packages are importable.')

# Optional, deliberately disabled installation command:
# !pip install numpy scipy astropy matplotlib photutils lenstronomy galight


## 2. Imports

In [ ]:
# INTERNAL - DO NOT EDIT FOR NORMAL USE
from __future__ import annotations
import copy
import csv
import dataclasses
import datetime as dt
import inspect
import json
import os
import platform
import sys
import traceback
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any

import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
from astropy.cosmology import Planck18
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.nddata import Cutout2D
from astropy.stats import sigma_clipped_stats
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
from photutils.aperture import CircularAperture
from photutils.centroids import centroid_com

# Galight 0.2.1 still calls np.int0; NumPy 2 removed that compatibility alias.
# This is the same narrow compatibility bridge used by the validated pipeline.
if not hasattr(np, 'int0'):
    np.int0 = np.intp

try:
    from galight.data_process import DataProcess
    from galight.fitting_process import FittingProcess
    from galight.fitting_specify import FittingSpecify
    from lenstronomy.ImSim.image_model import ImageModel
    from lenstronomy.Util import param_util
except ImportError as exc:
    raise RuntimeError(
        'This notebook requires galight and lenstronomy. Re-run the environment '
        'check above, install missing packages, restart, then Run All.'
    ) from exc


## 3. USER SETTINGS — EDIT ONLY THIS CELL

In [ ]:
# ======================================================================
# USER SETTINGS - EDIT ONLY THIS CELL
# ======================================================================

DEMO_MODE = True                       # First run: leave True and Run All.
RUN_MODE = 'SINGLE'                    # 'SINGLE' or optional 'BATCH'

# OBJECT
RA = None                              # degrees; required when DEMO_MODE=False
DEC = None                             # degrees; required when DEMO_MODE=False
SOURCE_REDSHIFT = 1.0
TARGET_REDSHIFT = 3.0

# NATIVE SOURCE DATA: used only to establish native morphology.
SOURCE_SCI_PATH = '...'
SOURCE_ERR_PATH = '...'
SOURCE_SEG_PATH = None
SOURCE_PSF_PATH = '...'
SOURCE_FILTER = 'F200W'

# TARGET BACKGROUND DATA: used only for target-z clean PSF and real injection.
# Set SAME_DATASET_BACKGROUND=True only if the same files genuinely serve both roles.
SAME_DATASET_BACKGROUND = False
TARGET_SCI_PATH = '...'
TARGET_ERR_PATH = '...'
TARGET_SEG_PATH = None
TARGET_PSF_PATH = '...'
TARGET_FILTER = 'F444W'

# PHOTOMETRY / SED SUPPORT
# Fluxes are F_nu in Jy, for example {'F200W': (2.0e-7, 2.0e-8)}.
# A lone source image does not predict arbitrary target-filter flux.
PHOTOMETRY = {}
SED_PATH = None                        # Optional two-column wavelength_um, Fnu_Jy file.

# MORPHOLOGY AND OUTPUT
MORPHOLOGY_MODE = 'FIT_NATIVE_FIRST'   # 'FIT_NATIVE_FIRST' or 'PARAMETRIC'
STRUCTURAL_MODE = 'SINGLE'             # 'SINGLE' or 'BULGE_DISK'
OUTPUT_DIR = './JWST_redshifting_output'

# PARAMETRIC mode only: Re is arcsec, PA is degrees east of +x, flux is Jy.
# For BULGE_DISK, provide components with roles 'disk' and 'bulge'.
PARAMETRIC_INPUT = {
    # 'components': [{'role': 'single', 'Re_arcsec': 0.15, 'n': 1.2,
    #                 'q': 0.7, 'PA_deg': 20., 'flux_jy': 2.0e-7}],
}

# Optional batch mode: CSV needs the settings-column names declared below.
BATCH_CATALOG_PATH = None
FORCE_RERUN_IDS = []


## 4. Advanced settings — normally leave unchanged

In [ ]:
# INTERNAL - DO NOT EDIT FOR NORMAL USE
NOISE_MODE = 'REAL_BACKGROUND_DETERMINISTIC'
# Other modes: REAL_BACKGROUND_SOURCE_POISSON or SYNTHETIC_BACKGROUND.
# The source-Poisson mode is refused without the metadata below.
COUNT_RATE_PER_DATA_UNIT = None         # count s^-1 per target internal data unit
EFFECTIVE_EXPOSURE_SECONDS = None
SOURCE_POISSON_SEED = 24680
# Used only with SYNTHETIC_BACKGROUND. Values are target internal data units
# (Jy/pixel after conversion); this is an explicitly uncorrelated model, not a
# substitute for a drizzled real-background mosaic.
SYNTHETIC_BACKGROUND_MODEL = None
# Example: {'shape': (101, 101), 'pixel_scale_arcsec': 0.03,
#           'background_mean': 0.0, 'background_rms': 1.0e-9, 'seed': 17}

# FITS extensions: None means first image HDU. Declare uncertainty semantics
# if AUTO cannot establish them: AUTO, ERR, RMS, VAR, IVAR, WHT.
SOURCE_SCI_EXT = None; SOURCE_ERR_EXT = None; SOURCE_SEG_EXT = None; SOURCE_PSF_EXT = None
TARGET_SCI_EXT = None; TARGET_ERR_EXT = None; TARGET_SEG_EXT = None; TARGET_PSF_EXT = None
SOURCE_UNCERTAINTY_KIND = 'AUTO'; TARGET_UNCERTAINTY_KIND = 'AUTO'
SOURCE_WEIGHT_TO_ERR_FACTOR = None; TARGET_WEIGHT_TO_ERR_FACTOR = None

# Set a separate target injection coordinate for a blank/context position.
# If omitted it defaults to RA/DEC and preflight warns if that position is segmented.
TARGET_INJECTION_RA = None
TARGET_INJECTION_DEC = None
CUTOUT_SIZE_ARCSEC = 3.0
PSF_PIXEL_SCALE_ARCSEC = None           # Scalar or {'source': ..., 'target': ...}
# Default mode is the easy empirical-PSF FITS path. Callable/PSFEx callables
# must return a 2-D array or (array, pixel_scale_arcsec) at the requested RA/Dec.
SOURCE_PSF_MODE = 'FITS'                 # FITS, CALLABLE, PSFEX_CALLABLE, WEBBPSF
TARGET_PSF_MODE = 'FITS'
PSF_CALLABLES = {'source': None, 'target': None}
WEBBPSF_SETTINGS = {'source': {}, 'target': {}}  # e.g. {'target': {'instrument':'NIRCam','fov_pixels':101}}
FILTER_PIVOT_UM = {}                    # Override/add a filter pivot wavelength safely.
SAME_REST_WAVELENGTH_TOLERANCE = 0.03
LUMINOSITY_EVOLUTION_FACTOR = 1.0       # Explicit multiplicative sensitivity only.
RESCALE_ERR_FROM_EMPIRICAL_NOISE = False
EMPIRICAL_NOISE_PATCH_PIXELS = 5
EMPIRICAL_NOISE_N_PATCHES = 150
N_FIT_REPEATS = 1
PSO_REPEATS = 1
RNG_SEED = 20260921
ALLOW_LABELED_SED_EXTRAPOLATION = False
SED_EXTRAPOLATION_LABEL = None

# This is a diagnostic warning, not a universal scientific B/T cut.
RESOLUTION_WARNING_RE_OVER_PSF = 0.5

# Batch CSV mapping. The default shares global data products and reads per-row
# ID, coordinate, redshift, and optional source paths/filters if supplied.
BATCH_COLUMN_MAP = {'case_id': 'id', 'ra': 'ra', 'dec': 'dec', 'z_source': 'z'}


## 5. Internal helper functions

In [ ]:
# INTERNAL - DO NOT EDIT FOR NORMAL USE

class PipelineFailure(RuntimeError):
    def __init__(self, status: str, code: str, message: str):
        super().__init__(f'[{status}:{code}] {message}')
        self.status, self.code, self.message = status, code, message


@dataclass
class Check:
    name: str
    state: str
    detail: str


@dataclass
class Dataset:
    role: str
    sci: np.ndarray
    err: np.ndarray | None
    seg: np.ndarray | None
    header: Any
    wcs: WCS
    pixel_scale_arcsec: float
    unit: str
    conversion_to_jy_per_pixel: float
    skycoord: SkyCoord
    position_xy: tuple[float, float]
    cutout: Cutout2D | None = None


@dataclass
class PSFInfo:
    role: str
    array: np.ndarray
    pixel_scale_arcsec: float
    centroid_xy: tuple[float, float]
    signed_total_before_normalization: float
    has_negative_wings: bool


COMMON_NIRCAM_PIVOT_UM = {
    'F070W': 0.704, 'F090W': 0.902, 'F115W': 1.154, 'F150W': 1.501,
    'F200W': 1.989, 'F277W': 2.776, 'F335M': 3.365, 'F356W': 3.568,
    'F410M': 4.083, 'F430M': 4.281, 'F444W': 4.421, 'F460M': 4.624,
    'F480M': 4.834,
}


def _now():
    return dt.datetime.now(dt.timezone.utc).isoformat()


def _plain(value):
    if isinstance(value, Path): return str(value)
    if isinstance(value, np.generic): return value.item()
    if isinstance(value, np.ndarray): return value.tolist()
    if isinstance(value, (list, tuple)): return [_plain(x) for x in value]
    if isinstance(value, set): return sorted(_plain(x) for x in value)
    if isinstance(value, dict): return {str(k): _plain(v) for k, v in value.items()}
    if callable(value): return {'callable': f'{getattr(value, "__module__", "unknown")}.{getattr(value, "__qualname__", getattr(value, "__name__", "callable"))}'}
    return value


def atomic_write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(_plain(payload), indent=2, sort_keys=True) + '\n')
    os.replace(tmp, path)


def append_log(output_dir, message):
    path = Path(output_dir) / 'run.log'
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a', encoding='utf-8') as handle:
        handle.write(f'{_now()} {message}\n')
        handle.flush(); os.fsync(handle.fileno())


def _get(user, name, default=None):
    return user[name] if name in user else default


def normalize_config(user):
    """Normalize the sole public settings cell into strictly separate roles."""
    demo = bool(_get(user, 'DEMO_MODE', False))
    same = bool(_get(user, 'SAME_DATASET_BACKGROUND', False))
    def role(role):
        upper = role.upper()
        return {
            'sci_path': _get(user, f'{upper}_SCI_PATH'),
            'err_path': _get(user, f'{upper}_ERR_PATH'),
            'seg_path': _get(user, f'{upper}_SEG_PATH'),
            'psf_path': _get(user, f'{upper}_PSF_PATH'),
            'psf_mode': str(_get(user, f'{upper}_PSF_MODE', 'FITS')).upper(),
            'filter': str(_get(user, f'{upper}_FILTER', '')).upper(),
            'sci_ext': _get(user, f'{upper}_SCI_EXT'),
            'err_ext': _get(user, f'{upper}_ERR_EXT'),
            'seg_ext': _get(user, f'{upper}_SEG_EXT'),
            'psf_ext': _get(user, f'{upper}_PSF_EXT'),
            'uncertainty_kind': str(_get(user, f'{upper}_UNCERTAINTY_KIND', 'AUTO')).upper(),
            'weight_to_err_factor': _get(user, f'{upper}_WEIGHT_TO_ERR_FACTOR'),
        }
    source, target = role('source'), role('target')
    if same:
        target = copy.deepcopy(source)
    cfg = {
        'demo_mode': demo, 'run_mode': str(_get(user, 'RUN_MODE', 'SINGLE')).upper(),
        'ra': _get(user, 'RA'), 'dec': _get(user, 'DEC'),
        'target_injection_ra': _get(user, 'TARGET_INJECTION_RA') or _get(user, 'RA'),
        'target_injection_dec': _get(user, 'TARGET_INJECTION_DEC') or _get(user, 'DEC'),
        'z_source': float(_get(user, 'SOURCE_REDSHIFT')),
        'z_target': float(_get(user, 'TARGET_REDSHIFT')),
        'source': source, 'target': target,
        'same_dataset_background': same,
        'photometry': _get(user, 'PHOTOMETRY', {}), 'sed_path': _get(user, 'SED_PATH'),
        'morphology_mode': str(_get(user, 'MORPHOLOGY_MODE', 'FIT_NATIVE_FIRST')).upper(),
        'structural_mode': str(_get(user, 'STRUCTURAL_MODE', 'SINGLE')).upper(),
        'parametric_input': _get(user, 'PARAMETRIC_INPUT', {}),
        'output_dir': str(_get(user, 'OUTPUT_DIR', './JWST_redshifting_output')),
        'noise_mode': str(_get(user, 'NOISE_MODE', 'REAL_BACKGROUND_DETERMINISTIC')).upper(),
        'count_rate_per_data_unit': _get(user, 'COUNT_RATE_PER_DATA_UNIT'),
        'effective_exposure_seconds': _get(user, 'EFFECTIVE_EXPOSURE_SECONDS'),
        'source_poisson_seed': int(_get(user, 'SOURCE_POISSON_SEED', 24680)),
        'synthetic_background_model': _get(user, 'SYNTHETIC_BACKGROUND_MODEL'),
        'cutout_size_arcsec': float(_get(user, 'CUTOUT_SIZE_ARCSEC', 3.0)),
        'psf_pixel_scale_arcsec': _get(user, 'PSF_PIXEL_SCALE_ARCSEC'),
        'psf_callables': dict(_get(user, 'PSF_CALLABLES', {})),
        'webbpsf_settings': dict(_get(user, 'WEBBPSF_SETTINGS', {})),
        'filter_pivot_um': dict(_get(user, 'FILTER_PIVOT_UM', {})),
        'same_rest_tolerance': float(_get(user, 'SAME_REST_WAVELENGTH_TOLERANCE', 0.03)),
        'luminosity_evolution_factor': float(_get(user, 'LUMINOSITY_EVOLUTION_FACTOR', 1.0)),
        'rescale_err': bool(_get(user, 'RESCALE_ERR_FROM_EMPIRICAL_NOISE', False)),
        'patch_pixels': int(_get(user, 'EMPIRICAL_NOISE_PATCH_PIXELS', 5)),
        'n_patches': int(_get(user, 'EMPIRICAL_NOISE_N_PATCHES', 150)),
        'n_fit_repeats': int(_get(user, 'N_FIT_REPEATS', 1)),
        'pso_repeats': int(_get(user, 'PSO_REPEATS', 1)),
        'rng_seed': int(_get(user, 'RNG_SEED', 20260921)),
        'allow_labeled_sed_extrapolation': bool(_get(user, 'ALLOW_LABELED_SED_EXTRAPOLATION', False)),
        'sed_extrapolation_label': _get(user, 'SED_EXTRAPOLATION_LABEL'),
        'resolution_warning_re_over_psf': float(_get(user, 'RESOLUTION_WARNING_RE_OVER_PSF', 0.5)),
        'batch_catalog_path': _get(user, 'BATCH_CATALOG_PATH'),
        'batch_column_map': dict(_get(user, 'BATCH_COLUMN_MAP', {})),
        'force_rerun_ids': {str(x) for x in _get(user, 'FORCE_RERUN_IDS', [])},
    }
    if cfg['run_mode'] not in {'SINGLE', 'BATCH'}:
        raise PipelineFailure('TERMINAL_ERROR', 'INVALID_RUN_MODE', 'RUN_MODE must be SINGLE or BATCH.')
    if cfg['morphology_mode'] not in {'FIT_NATIVE_FIRST', 'PARAMETRIC'}:
        raise PipelineFailure('TERMINAL_ERROR', 'INVALID_MORPHOLOGY_MODE', 'Use FIT_NATIVE_FIRST or PARAMETRIC.')
    if cfg['structural_mode'] not in {'SINGLE', 'BULGE_DISK'}:
        raise PipelineFailure('TERMINAL_ERROR', 'INVALID_STRUCTURAL_MODE', 'Use SINGLE or BULGE_DISK.')
    if cfg['noise_mode'] not in {'REAL_BACKGROUND_DETERMINISTIC', 'REAL_BACKGROUND_SOURCE_POISSON', 'SYNTHETIC_BACKGROUND'}:
        raise PipelineFailure('TERMINAL_ERROR', 'INVALID_NOISE_MODE', 'Unknown NOISE_MODE.')
    for role_name in ('source', 'target'):
        if cfg[role_name]['psf_mode'] not in {'FITS', 'CALLABLE', 'PSFEX_CALLABLE', 'WEBBPSF'}:
            raise PipelineFailure('TERMINAL_ERROR', 'INVALID_PSF_MODE', f'{role_name} PSF mode is unsupported.')
    if not demo and (cfg['ra'] is None or cfg['dec'] is None):
        raise PipelineFailure('TERMINAL_ERROR', 'MISSING_COORDINATES', 'Set RA and DEC in degrees.')
    if cfg['z_source'] <= 0 or cfg['z_target'] <= 0:
        raise PipelineFailure('TERMINAL_ERROR', 'INVALID_REDSHIFT', 'Both redshifts must be finite and positive.')
    if not demo:
        for role_name in ('target',):
            target_keys = ('psf_path', 'filter') if cfg['noise_mode'] == 'SYNTHETIC_BACKGROUND' else ('sci_path', 'err_path', 'psf_path', 'filter')
            for key in target_keys:
                if key == 'psf_path' and cfg[role_name]['psf_mode'] != 'FITS':
                    continue
                if not cfg[role_name][key] or cfg[role_name][key] == '...':
                    raise PipelineFailure('TERMINAL_ERROR', 'MISSING_TARGET_INPUT', f'TARGET {key} is required.')
        if cfg['morphology_mode'] == 'FIT_NATIVE_FIRST':
            for key in ('sci_path', 'err_path', 'psf_path', 'filter'):
                if key == 'psf_path' and cfg['source']['psf_mode'] != 'FITS':
                    continue
                if not cfg['source'][key] or cfg['source'][key] == '...':
                    raise PipelineFailure('TERMINAL_ERROR', 'MISSING_SOURCE_INPUT', f'SOURCE {key} is required for FIT_NATIVE_FIRST.')
    return cfg


def filter_pivot_um(name, config):
    name = str(name).upper()
    table = {**COMMON_NIRCAM_PIVOT_UM, **{str(k).upper(): float(v) for k, v in config['filter_pivot_um'].items()}}
    if name not in table:
        raise PipelineFailure('TERMINAL_ERROR', 'UNKNOWN_FILTER_PIVOT',
                              f'No pivot wavelength for {name}. Add FILTER_PIVOT_UM={{"{name}": value_in_um}}.')
    return table[name]


def _first_image_hdu(path, extension):
    try:
        with fits.open(path, memmap=False) as hdul:
            if extension is not None:
                hdu = hdul[extension]
                if hdu.data is None: raise ValueError('selected extension has no image data')
                return np.array(hdu.data, dtype=float), hdu.header.copy()
            for hdu in hdul:
                if hdu.data is not None and np.ndim(hdu.data) == 2:
                    return np.array(hdu.data, dtype=float), hdu.header.copy()
    except FileNotFoundError as exc:
        raise PipelineFailure('TERMINAL_ERROR', 'MISSING_FITS', f'Cannot find FITS file: {path}') from exc
    except Exception as exc:
        raise PipelineFailure('TERMINAL_ERROR', 'UNREADABLE_FITS', f'Cannot read {path}: {exc}') from exc
    raise PipelineFailure('TERMINAL_ERROR', 'NO_2D_IMAGE', f'No 2-D image HDU in {path}.')


def _pixel_scale_arcsec(wcs, header):
    try:
        scales = proj_plane_pixel_scales(wcs) * 3600.0
        scale = float(np.sqrt(scales[0] * scales[1]))
    except Exception:
        scale = float('nan')
    if not np.isfinite(scale) or scale <= 0:
        raise PipelineFailure('TERMINAL_ERROR', 'INVALID_WCS_SCALE',
                              'A valid celestial WCS/pixel scale is required; supply a FITS WCS.')
    return scale


def _unit_factor_to_jy_per_pixel(unit_text, pixel_scale_arcsec):
    raw = str(unit_text or '').strip().lower().replace(' ', '')
    pixel_sr = (pixel_scale_arcsec * u.arcsec) ** 2
    pixel_sr = pixel_sr.to_value(u.sr)
    if raw in {'mjy/sr', 'mjy/steradian', 'mjysr-1', 'mjy/sr.'}:
        return 1e6 * pixel_sr, 'MJy/sr'
    if raw in {'jy', 'jy/pixel', 'jy/pix'}:
        return 1.0, 'Jy/pixel'
    if raw in {'ujy', 'µjy', 'microjy', 'ujy/pixel', 'µjy/pixel'}:
        return 1e-6, 'uJy/pixel'
    if raw in {'njy', 'njy/pixel'}:
        return 1e-9, 'nJy/pixel'
    raise PipelineFailure('TERMINAL_ERROR', 'UNKNOWN_SCI_UNIT',
                          f'Unsupported BUNIT={unit_text!r}. Supported public defaults are MJy/sr, Jy/pixel, uJy/pixel, nJy/pixel. Provide a calibrated conversion before continuing.')


def _err_from_map(values, kind, header, weight_to_err_factor):
    kind = str(kind).upper()
    if kind == 'AUTO':
        extname = str(header.get('EXTNAME', '')).upper()
        kind = 'VAR' if 'VAR' in extname else 'IVAR' if 'IVAR' in extname else 'WHT' if 'WHT' in extname else 'ERR'
    values = np.asarray(values, float)
    if kind in {'ERR', 'RMS'}:
        err = values
    elif kind == 'VAR':
        err = np.sqrt(np.where(values >= 0, values, np.nan))
    elif kind == 'IVAR':
        err = np.sqrt(np.where(values > 0, 1.0 / values, np.nan))
    elif kind == 'WHT':
        if weight_to_err_factor is None:
            raise PipelineFailure('TERMINAL_ERROR', 'AMBIGUOUS_WEIGHT_MAP',
                                  'A WHT map is not automatically an ERR map. Set TARGET_WEIGHT_TO_ERR_FACTOR (or source equivalent) with documented units.')
        err = float(weight_to_err_factor) * np.sqrt(np.where(values > 0, 1.0 / values, np.nan))
    else:
        raise PipelineFailure('TERMINAL_ERROR', 'UNKNOWN_UNCERTAINTY_KIND', f'Unsupported uncertainty kind {kind}.')
    return err, kind


def _cutout_for_role(role, config, sci, err, seg, header):
    try:
        wcs = WCS(header).celestial
        if not wcs.has_celestial: raise ValueError('no celestial axes')
    except Exception as exc:
        raise PipelineFailure('TERMINAL_ERROR', 'INVALID_WCS', f'{role} SCI has no usable celestial WCS: {exc}') from exc
    scale = _pixel_scale_arcsec(wcs, header)
    ra = config['ra'] if role == 'source' else config['target_injection_ra']
    dec = config['dec'] if role == 'source' else config['target_injection_dec']
    coord = SkyCoord(float(ra) * u.deg, float(dec) * u.deg)
    x, y = wcs.world_to_pixel(coord)
    if not (0 <= x < sci.shape[1] and 0 <= y < sci.shape[0]):
        raise PipelineFailure('TERMINAL_ERROR', 'OUTSIDE_MOSAIC', f'{role} coordinate is outside its SCI mosaic.')
    size = max(25, int(np.ceil(config['cutout_size_arcsec'] / scale)))
    if size % 2 == 0: size += 1
    try:
        cut = Cutout2D(sci, coord, (size, size), wcs=wcs, mode='strict', copy=True)
        err_cut = Cutout2D(err, coord, (size, size), wcs=wcs, mode='strict', copy=True).data if err is not None else None
        seg_cut = Cutout2D(seg, coord, (size, size), wcs=wcs, mode='strict', copy=True).data if seg is not None else None
    except Exception as exc:
        raise PipelineFailure('TERMINAL_ERROR', 'INSUFFICIENT_VALID_PIXELS', f'{role} cutout cannot be made: {exc}') from exc
    return cut.data, err_cut, seg_cut, header, cut.wcs, scale, coord, (size // 2, size // 2), cut


def load_dataset(role, config):
    spec = config[role]
    sci, header = _first_image_hdu(spec['sci_path'], spec['sci_ext'])
    if not np.all(np.isfinite(sci) | np.isnan(sci)):
        raise PipelineFailure('TERMINAL_ERROR', 'INVALID_SCI', f'{role} SCI has invalid values.')
    wcs_for_scale = WCS(header).celestial
    scale = _pixel_scale_arcsec(wcs_for_scale, header)
    factor, unit_label = _unit_factor_to_jy_per_pixel(header.get('BUNIT'), scale)
    err = None
    if spec['err_path']:
        raw_err, err_header = _first_image_hdu(spec['err_path'], spec['err_ext'])
        if raw_err.shape != sci.shape:
            raise PipelineFailure('TERMINAL_ERROR', 'ERR_SHAPE_MISMATCH', f'{role} ERR shape differs from SCI.')
        err, _ = _err_from_map(raw_err, spec['uncertainty_kind'], err_header, spec['weight_to_err_factor'])
    seg = None
    if spec['seg_path']:
        seg, _ = _first_image_hdu(spec['seg_path'], spec['seg_ext'])
        if seg.shape != sci.shape:
            raise PipelineFailure('TERMINAL_ERROR', 'SEG_SHAPE_MISMATCH', f'{role} segmentation shape differs from SCI.')
        seg = np.asarray(seg, int)
    data = _cutout_for_role(role, config, sci * factor, None if err is None else err * factor, seg, header)
    return Dataset(role, *data[:3], data[3], data[4], data[5], unit_label, factor, data[6], data[7], data[8])


def make_synthetic_target(config):
    """Construct only the user-declared synthetic background; no hidden noise terms."""
    model = config.get('synthetic_background_model')
    required = {'shape', 'pixel_scale_arcsec', 'background_mean', 'background_rms', 'seed'}
    if not isinstance(model, dict) or not required.issubset(model):
        raise PipelineFailure('TERMINAL_ERROR', 'MISSING_SYNTHETIC_BACKGROUND_MODEL',
                              'SYNTHETIC_BACKGROUND needs SYNTHETIC_BACKGROUND_MODEL with shape, pixel_scale_arcsec, background_mean, background_rms, and seed.')
    shape = tuple(int(x) for x in model['shape'])
    scale = float(model['pixel_scale_arcsec']); rms = float(model['background_rms'])
    if len(shape) != 2 or min(shape) < 25 or shape[0] != shape[1] or scale <= 0 or rms <= 0:
        raise PipelineFailure('TERMINAL_ERROR', 'INVALID_SYNTHETIC_BACKGROUND_MODEL',
                              'Synthetic shape must be square and at least 25 pixels; pixel scale and RMS must be positive.')
    rng = np.random.default_rng(int(model['seed']))
    sci = float(model['background_mean']) + rng.normal(0., rms, size=shape)
    err = np.full(shape, rms)
    header = make_minimal_wcs_header(shape[0], scale); header['BUNIT'] = 'Jy/pixel'
    wcs = WCS(header).celestial
    sky = SkyCoord(0. * u.deg, 0. * u.deg)
    return Dataset('target', sci, err, None, header, wcs, scale, 'Jy/pixel', 1., sky,
                   ((shape[1]-1)/2, (shape[0]-1)/2), None)


def _resolve_psf_scale(role, config, header):
    setting = config['psf_pixel_scale_arcsec']
    if isinstance(setting, dict): setting = setting.get(role)
    if setting is not None:
        return float(setting)
    try:
        return _pixel_scale_arcsec(WCS(header).celestial, header)
    except PipelineFailure:
        raise PipelineFailure('TERMINAL_ERROR', 'MISSING_PSF_PIXEL_SCALE',
                              f'{role} PSF has no usable WCS. Set PSF_PIXEL_SCALE_ARCSEC for this PSF.')


def prepare_psf(role, config, dataset):
    spec = config[role]
    mode = spec['psf_mode']
    if mode == 'FITS':
        psf, header = _first_image_hdu(spec['psf_path'], spec['psf_ext'])
    elif mode in {'CALLABLE', 'PSFEX_CALLABLE'}:
        evaluator = config['psf_callables'].get(role)
        if not callable(evaluator):
            raise PipelineFailure('TERMINAL_ERROR', 'MISSING_PSF_CALLABLE',
                                  f'Set PSF_CALLABLES["{role}"] for {mode}; PSFEx coordinate conventions are release-specific and cannot be guessed.')
        try:
            supplied = evaluator(ra_deg=dataset.skycoord.ra.deg, dec_deg=dataset.skycoord.dec.deg, role=role)
        except TypeError:
            supplied = evaluator(dataset.skycoord.ra.deg, dataset.skycoord.dec.deg)
        if isinstance(supplied, tuple):
            psf, supplied_scale = supplied
            scale_setting = config['psf_pixel_scale_arcsec']
            if isinstance(scale_setting, dict): scale_setting = dict(scale_setting, **{role: supplied_scale})
            else: scale_setting = {role: supplied_scale}
            config['psf_pixel_scale_arcsec'] = scale_setting
        else:
            psf = supplied
        header = fits.Header()
    else:  # WEBBPSF
        settings = config['webbpsf_settings'].get(role, {})
        try:
            import webbpsf
        except ImportError as exc:
            raise PipelineFailure('TERMINAL_ERROR', 'MISSING_WEBBPSF', 'Install webbpsf or use an empirical PSF FITS image.') from exc
        instrument_name = settings.get('instrument', 'NIRCam')
        try:
            instrument = getattr(webbpsf, instrument_name)()
            instrument.filter = settings.get('filter', spec['filter'])
            output = instrument.calc_psf(fov_pixels=int(settings.get('fov_pixels', 101)))
            psf = np.asarray(output[0].data, float)
            header = output[0].header.copy()
        except Exception as exc:
            raise PipelineFailure('TERMINAL_ERROR', 'WEBBPSF_FAILURE', f'WebbPSF could not create {role} PSF: {exc}') from exc
    if psf.ndim != 2 or min(psf.shape) < 3 or not np.all(np.isfinite(psf)):
        raise PipelineFailure('TERMINAL_ERROR', 'INVALID_PSF', f'{role} PSF must be finite, 2-D, and at least 3x3.')
    signed_total = float(psf.sum())
    if not np.isfinite(signed_total) or signed_total <= 0:
        raise PipelineFailure('TERMINAL_ERROR', 'INVALID_PSF_NORMALIZATION', f'{role} signed PSF sum must be positive; it is {signed_total}.')
    scale = _resolve_psf_scale(role, config, header)
    if not np.isclose(scale, dataset.pixel_scale_arcsec, rtol=0.03, atol=1e-6):
        raise PipelineFailure('TERMINAL_ERROR', 'PSF_SCIENCE_SCALE_MISMATCH',
                              f'{role} PSF scale {scale:.5g} differs from SCI scale {dataset.pixel_scale_arcsec:.5g}. Resample PSF with a documented method first.')
    normalized = psf / signed_total  # Signed negative wings are intentionally preserved.
    centroid = centroid_com(np.clip(normalized, 0, None))
    geometric = ((normalized.shape[1] - 1) / 2, (normalized.shape[0] - 1) / 2)
    offset = float(np.hypot(centroid[0] - geometric[0], centroid[1] - geometric[1]))
    if offset > 1.0:
        raise PipelineFailure('TERMINAL_ERROR', 'OFFCENTER_PSF', f'{role} PSF centroid is {offset:.2f} pixels from its geometric center.')
    return PSFInfo(role, normalized, scale, tuple(map(float, centroid)), signed_total, bool(np.any(normalized < 0)))


def make_minimal_wcs_header(npix, pixel_scale_arcsec):
    header = fits.Header()
    header['NAXIS'] = 2; header['NAXIS1'] = int(npix); header['NAXIS2'] = int(npix)
    header['CTYPE1'] = 'RA---TAN'; header['CTYPE2'] = 'DEC--TAN'
    header['CRPIX1'] = (npix + 1) / 2; header['CRPIX2'] = (npix + 1) / 2
    header['CRVAL1'] = 0.; header['CRVAL2'] = 0.
    header['CDELT1'] = -pixel_scale_arcsec / 3600.; header['CDELT2'] = pixel_scale_arcsec / 3600.
    return header


def _phi_q_to_e(phi_rad, q):
    q = float(np.clip(q, 0.05, 1.0))
    return param_util.phi_q2_ellipticity(float(phi_rad), q)


def build_source_params(components, npix, pixel_scale_arcsec, initial_offset_arcsec=(0., 0.)):
    re_max = (npix - 1) * pixel_scale_arcsec / 2
    init = []; sigma = []; fixed = []; lower = []; upper = []
    for component in components:
        n = float(component.get('n_sersic', component.get('n', 1.0)))
        re = float(component.get('R_sersic', component.get('Re_arcsec', 0.15)))
        q = float(component.get('q', 0.7))
        phi = float(component.get('phi_G', np.deg2rad(component.get('PA_deg', 0.))))
        e1, e2 = _phi_q_to_e(phi, q)
        this_init = {'R_sersic': float(np.clip(re, 0.01, 0.8 * re_max)), 'n_sersic': n,
                     'e1': e1, 'e2': e2, 'center_x': float(component.get('center_x', initial_offset_arcsec[0])),
                     'center_y': float(component.get('center_y', initial_offset_arcsec[1]))}
        this_fixed = {}
        if component.get('fixed_n') is not None:
            this_fixed['n_sersic'] = float(component['fixed_n']); this_init['n_sersic'] = float(component['fixed_n'])
        init.append(this_init)
        sigma.append({'R_sersic': max(0.03, 0.3 * re), 'n_sersic': 1.0, 'e1': 0.1, 'e2': 0.1,
                      'center_x': pixel_scale_arcsec, 'center_y': pixel_scale_arcsec})
        fixed.append(this_fixed)
        lower.append({'R_sersic': 0.01, 'n_sersic': 0.3, 'e1': -0.58, 'e2': -0.58,
                      'center_x': -2 * pixel_scale_arcsec, 'center_y': -2 * pixel_scale_arcsec})
        upper.append({'R_sersic': re_max, 'n_sersic': 9.0, 'e1': 0.58, 'e2': 0.58,
                      'center_x': 2 * pixel_scale_arcsec, 'center_y': 2 * pixel_scale_arcsec})
    return [init, sigma, fixed, lower, upper]


def build_galight_data_process(image, err, seg, psf, pixel_scale_arcsec):
    image = np.asarray(image, float); err = np.asarray(err, float)
    if image.shape != err.shape or image.ndim != 2 or image.shape[0] != image.shape[1]:
        raise PipelineFailure('TERMINAL_ERROR', 'INVALID_FIT_ARRAYS', 'Fit image and ERR must be same-size square 2-D arrays.')
    valid = np.isfinite(image) & np.isfinite(err) & (err > 0)
    if valid.sum() < 0.5 * image.size:
        raise PipelineFailure('TERMINAL_ERROR', 'INSUFFICIENT_VALID_PIXELS', 'Fewer than half of fit pixels have finite positive ERR.')
    safe_image = np.where(np.isfinite(image), image, 0.)
    huge = np.nanmedian(err[valid]) * 1e6
    safe_err = np.where(valid, err, huge)
    if seg is None:
        likelihood_mask = valid.astype(float)
    else:
        seg = np.asarray(seg, int)
        cy, cx = np.array(seg.shape) // 2
        target_label = int(seg[cy, cx])
        other = (seg > 0) & ((target_label == 0) | (seg != target_label))
        likelihood_mask = (valid & ~other).astype(float)
    dp = DataProcess(fov_image=safe_image, target_pos=(image.shape[1]//2, image.shape[0]//2),
                     pos_type='pixel', header=make_minimal_wcs_header(image.shape[0], pixel_scale_arcsec),
                     fov_noise_map=safe_err, rm_bkglight=False, if_plot=False, zp=27.0)
    dp.target_stamp = safe_image; dp.noise_map = safe_err; dp.target_mask = likelihood_mask
    dp.segm_deblend = np.zeros_like(image, int) if seg is None else seg
    dp.tbl = None; dp.apertures = []; dp.mask_apertures = []; dp.radius = image.shape[0] // 2
    dp.PSF_list = [np.asarray(psf, float)]; dp.psf_id_for_fitting = 0
    return dp, likelihood_mask


def _safe_image_model(fit_spec):
    candidates = {'data_class': fit_spec.data_class, 'psf_class': fit_spec.psf_class,
                  'lens_light_model_class': fit_spec.lightModel, 'point_source_class': fit_spec.pointSource,
                  'kwargs_numerics': fit_spec.kwargs_numerics}
    if 'likelihood_mask' in inspect.signature(ImageModel.__init__).parameters:
        candidates['likelihood_mask'] = fit_spec.kwargs_likelihood['image_likelihood_mask_list'][0]
    accepted = inspect.signature(ImageModel.__init__).parameters
    return ImageModel(**{k: v for k, v in candidates.items() if k in accepted})


def _image_call(image_model, kwargs_lens_light):
    candidates = {'kwargs_lens': None, 'kwargs_source': None, 'kwargs_lens_light': kwargs_lens_light,
                  'kwargs_ps': None, 'kwargs_extinction': None, 'kwargs_special': None,
                  'unconvolved': False, 'source_add': False, 'lens_light_add': True, 'point_source_add': False}
    accepted = inspect.signature(image_model.image).parameters
    return image_model.image(**{k: v for k, v in candidates.items() if k in accepted})


def render_lenstronomy(components, shape, pixel_scale_arcsec, psf, component_fluxes=None):
    """Render all structural truth through Lenstronomy ImageModel, never an analytic substitute."""
    blank = np.zeros(shape, float); err = np.ones(shape, float)
    dp, _ = build_galight_data_process(blank, err, np.zeros(shape, int), psf, pixel_scale_arcsec)
    source_params = build_source_params(components, shape[0], pixel_scale_arcsec)
    fit_spec = FittingSpecify(dp, sersic_major_axis=True)
    fit_spec.prepare_fitting_seq(supersampling_factor=2, psf_data=psf,
                                 extend_source_model=['SERSIC_ELLIPSE'] * len(components),
                                 point_source_num=0, source_params=source_params, condition=None, mpi=False)
    model = _safe_image_model(fit_spec)
    rendered = []
    for i, component in enumerate(components):
        kwargs = copy.deepcopy(source_params[0])
        for item in kwargs: item['amp'] = 0.0
        kwargs[i]['amp'] = float(component.get('amp', 1.0))
        one = np.asarray(_image_call(model, kwargs), float)
        if not np.all(np.isfinite(one)) or one.sum() <= 0:
            raise PipelineFailure('TERMINAL_ERROR', 'NONFINITE_RENDER', 'Lenstronomy produced invalid model pixels.')
        rendered.append(one)
    if component_fluxes is not None:
        rendered = [im * (float(flux) / im.sum()) for im, flux in zip(rendered, component_fluxes)]
    return np.sum(rendered, axis=0), rendered


def run_galight_fit(image, err, seg, psf, pixel_scale_arcsec, components, pso_repeats=1, savename=None):
    dp, mask = build_galight_data_process(image, err, seg, psf, pixel_scale_arcsec)
    source_params = build_source_params(components, image.shape[0], pixel_scale_arcsec)
    fit_spec = FittingSpecify(dp, sersic_major_axis=True)
    fit_spec.prepare_fitting_seq(supersampling_factor=2, psf_data=psf,
                                 extend_source_model=['SERSIC_ELLIPSE'] * len(components),
                                 point_source_num=0, source_params=source_params, condition=None, mpi=False)
    run = FittingProcess(fit_spec, savename=str(savename or 'galight_fit'))
    run.run(algorithm_list=['PSO'] * int(max(1, pso_repeats)), fitting_level='norm', threadCount=1)
    recovered = []
    for row in run.final_result_galaxy:
        recovered.append({k: float(v) if isinstance(v, (int, float, np.number)) else v for k, v in row.items()})
    lower, upper = source_params[3], source_params[4]
    hits = []
    for i, row in enumerate(recovered):
        for name in ('R_sersic', 'n_sersic', 'e1', 'e2', 'center_x', 'center_y'):
            if name in row and name in lower[i] and np.isclose(row[name], lower[i][name], rtol=0, atol=1e-4): hits.append(f'{i}:{name}:lower')
            if name in row and name in upper[i] and np.isclose(row[name], upper[i][name], rtol=0, atol=1e-4): hits.append(f'{i}:{name}:upper')
    model_image = np.sum(np.asarray(run.image_host_list), axis=0)
    residual = np.asarray(image) - model_image
    return {'components': recovered, 'model_image': model_image, 'residual': residual,
            'chisq': float(run.reduced_Chisq), 'bound_hits': hits, 'mask': mask,
            'source_result': copy.deepcopy(run.source_result)}


def _components_for_fit(structural_mode, image, pixel_scale_arcsec):
    radius = max(0.06, 0.12 * image.shape[0] * pixel_scale_arcsec)
    if structural_mode == 'BULGE_DISK':
        return [{'role': 'disk', 'Re_arcsec': radius, 'n': 1., 'q': .7, 'PA_deg': 0., 'fixed_n': 1.},
                {'role': 'bulge', 'Re_arcsec': radius / 3., 'n': 4., 'q': .8, 'PA_deg': 0., 'fixed_n': 4.}]
    return [{'role': 'single', 'Re_arcsec': radius, 'n': 1.5, 'q': .7, 'PA_deg': 0.}]


def _component_fluxes(result):
    return [float(row.get('flux_within_frame', np.nan)) for row in result['components']]


def _as_model_components(result):
    out = []
    for row in result['source_result']:
        out.append({'R_sersic': float(row['R_sersic']), 'n_sersic': float(row['n_sersic']),
                    'q': float(param_util.ellipticity2phi_q(row['e1'], row['e2'])[1]),
                    'phi_G': float(param_util.ellipticity2phi_q(row['e1'], row['e2'])[0]),
                    'center_x': float(row.get('center_x', 0.)), 'center_y': float(row.get('center_y', 0.)),
                    'amp': float(row.get('amp', 1.))})
    return out


def _photometry_plan(config):
    ls = filter_pivot_um(config['source']['filter'], config)
    lt = filter_pivot_um(config['target']['filter'], config)
    required_source_lambda = lt * (1 + config['z_source']) / (1 + config['z_target'])
    rest_source, rest_target = ls / (1 + config['z_source']), lt / (1 + config['z_target'])
    ratio = ((1 + config['z_target']) / (1 + config['z_source']) *
             (Planck18.luminosity_distance(config['z_source']).value /
              Planck18.luminosity_distance(config['z_target']).value) ** 2)
    plan = {'source_pivot_um': ls, 'target_pivot_um': lt, 'required_source_lambda_um': required_source_lambda,
            'source_rest_um': rest_source, 'target_rest_um': rest_target,
            'cosmological_fnu_ratio': float(ratio), 'luminosity_evolution_factor': config['luminosity_evolution_factor']}
    if abs(rest_source / rest_target - 1) <= config['same_rest_tolerance']:
        plan.update({'mode': 'same_rest_wavelength', 'source_equivalent_fnu_jy': None,
                     'target_fnu_jy': None, 'support': 'PASS: source/target filters match rest wavelength within configured tolerance.'})
        return plan
    photometry = {str(k).upper(): tuple(v) for k, v in config['photometry'].items()}
    if config['sed_path']:
        try:
            table = np.loadtxt(config['sed_path'])
            wave, fnu = np.asarray(table[:, 0], float), np.asarray(table[:, 1], float)
        except Exception as exc:
            raise PipelineFailure('TERMINAL_ERROR', 'UNREADABLE_SED', f'Cannot read SED_PATH: {exc}') from exc
    else:
        usable = [(filter_pivot_um(name, config), float(value[0])) for name, value in photometry.items()]
        if len(usable) < 2:
            raise PipelineFailure('TERMINAL_ERROR', 'UNSUPPORTED_WAVELENGTH',
                                  f'Target {config["target"]["filter"]} at z={config["z_target"]} requires source-frame {required_source_lambda:.3f} um. Supply bracketing PHOTOMETRY or SED_PATH; one native image is insufficient.')
        wave, fnu = map(np.asarray, zip(*sorted(usable)))
    if required_source_lambda < wave.min() or required_source_lambda > wave.max():
        if not (config['allow_labeled_sed_extrapolation'] and config['sed_extrapolation_label']):
            raise PipelineFailure('TERMINAL_ERROR', 'UNSUPPORTED_WAVELENGTH',
                                  f'Required source wavelength {required_source_lambda:.3f} um lies outside available {wave.min():.3f}–{wave.max():.3f} um support. Do not silently extrapolate; supply an SED or explicit labeled method.')
        support = 'WARNING: explicit user-labeled SED extrapolation.'
    else:
        support = 'PASS: target rest wavelength is bracketed by supplied photometry/SED.'
    f_source = float(np.interp(required_source_lambda, wave, fnu))
    plan.update({'mode': 'interpolated_or_labeled_sed', 'source_equivalent_fnu_jy': f_source,
                 'target_fnu_jy': f_source * ratio * config['luminosity_evolution_factor'], 'support': support})
    return plan


def empirical_noise_audit(dataset, config, protected_radius_pix=None):
    if dataset.err is None:
        raise PipelineFailure('TERMINAL_ERROR', 'MISSING_ERR', f'{dataset.role} ERR/RMS map is required for real-background injection.')
    rng = np.random.default_rng(config['rng_seed'] + (0 if dataset.role == 'source' else 1))
    image, err = dataset.sci, dataset.err
    valid = np.isfinite(image) & np.isfinite(err) & (err > 0)
    cy, cx = np.array(image.shape) // 2
    yy, xx = np.indices(image.shape)
    protected_radius_pix = protected_radius_pix or max(4, config['patch_pixels'])
    protected = (xx - cx) ** 2 + (yy - cy) ** 2 <= protected_radius_pix ** 2
    if dataset.seg is not None:
        blank = valid & (dataset.seg == 0) & ~protected
        seg_warning = None
    else:
        blank = valid & ~protected
        seg_warning = 'WARNING: no external segmentation; blank-sky uses protected-source masking plus sigma-clipped robust patches, not all unmasked pixels.'
    half = max(1, config['patch_pixels'] // 2)
    candidates = np.argwhere(blank)
    samples = []
    for y, x in candidates[rng.permutation(len(candidates))[:max(len(candidates), 1)]]:
        patch = image[max(0,y-half):min(image.shape[0],y+half+1), max(0,x-half):min(image.shape[1],x+half+1)]
        patch_blank = blank[max(0,y-half):min(image.shape[0],y+half+1), max(0,x-half):min(image.shape[1],x+half+1)]
        if patch.shape == (2*half+1, 2*half+1) and patch_blank.mean() > .8:
            _, _, std = sigma_clipped_stats(patch, sigma=3.)
            if np.isfinite(std) and std > 0: samples.append(float(std))
            if len(samples) >= config['n_patches']: break
    if len(samples) < 10:
        raise PipelineFailure('TERMINAL_ERROR', 'NO_BLANK_SKY', 'Fewer than ten robust blank-sky patches were available; provide segmentation or a cleaner target location.')
    empirical = float(np.median(samples))
    expected = float(np.nanmedian(err[blank]))
    ratio = empirical / expected
    state = 'PASS' if .8 <= ratio <= 1.25 else 'WARNING'
    return {'state': state, 'empirical_rms': empirical, 'median_err': expected, 'empirical_to_err': ratio,
            'n_patches': len(samples), 'segmentation_warning': seg_warning, 'samples': samples}


def inject_real_background(target, model, config):
    if target.err is None:
        raise PipelineFailure('TERMINAL_ERROR', 'MISSING_ERR', 'TARGET_ERR_PATH is required for real-background injection.')
    if config['noise_mode'] == 'REAL_BACKGROUND_DETERMINISTIC':
        return {'image': target.sci + model, 'err': target.err.copy(), 'background_noise_draws': 0,
                'source_poisson_added': False, 'noise_statement': 'Deterministic injected source; existing target background/read/correlated noise retained with no second realization.'}
    if config['noise_mode'] == 'REAL_BACKGROUND_SOURCE_POISSON':
        rate = config['count_rate_per_data_unit']; exposure = config['effective_exposure_seconds']
        if rate is None or exposure is None or float(rate) <= 0 or float(exposure) <= 0:
            raise PipelineFailure('TERMINAL_ERROR', 'MISSING_SOURCE_POISSON_METADATA',
                                  'Source-Poisson mode needs valid COUNT_RATE_PER_DATA_UNIT and EFFECTIVE_EXPOSURE_SECONDS; MJy/sr alone is not a count conversion.')
        gain = float(rate) * float(exposure)
        expected_counts = np.clip(model * gain, 0, None)
        rng = np.random.default_rng(config['source_poisson_seed'])
        realization = (rng.poisson(expected_counts) - expected_counts) / gain
        variance = expected_counts / gain**2
        return {'image': target.sci + model + realization, 'err': np.sqrt(target.err**2 + variance),
                'background_noise_draws': 0, 'source_poisson_added': True,
                'noise_statement': 'Only injected-source Poisson realization/variance added; pre-existing target background noise was not redrawn.'}
    if config['noise_mode'] == 'SYNTHETIC_BACKGROUND':
        return {'image': target.sci + model, 'err': target.err.copy(), 'background_noise_draws': 1,
                'source_poisson_added': False,
                'noise_statement': 'One user-declared synthetic Gaussian background realization was created before injection; no additional sky/read realization was added during injection.'}
    raise PipelineFailure('TERMINAL_ERROR', 'INVALID_NOISE_MODE', 'Unknown noise mode at injection.')


def _quality_summary(fit, psf, config):
    comps = fit['components']; fluxes = np.asarray(_component_fluxes(fit), float)
    total = float(fluxes.sum()) if np.all(np.isfinite(fluxes)) else np.nan
    out = {'chisq': fit['chisq'], 'bound_hits': fit['bound_hits'], 'n_components': len(comps),
           'components': comps, 'fluxes': fluxes.tolist(), 'total_model_flux': total}
    if len(comps) == 1:
        out.update({'single_n': float(comps[0]['n_sersic']), 'single_re_arcsec': float(comps[0]['R_sersic']),
                    'single_q': float(comps[0]['q']), 'bt': np.nan, 'bt_identifiability': 'NOT_APPLICABLE_SINGLE_COMPONENT',
                    'disk_classification': 'NOT_APPLICABLE_SINGLE_COMPONENT'})
    elif len(comps) == 2:
        bt = float(fluxes[1] / total) if np.isfinite(total) and total > 0 else np.nan
        ratios = [float(c['R_sersic']) / (psf.pixel_scale_arcsec * max(1, psf.array.shape) / 2) for c in comps]
        warning = any(r < config['resolution_warning_re_over_psf'] for r in ratios) or bool(fit['bound_hits'])
        out.update({'bt': bt, 'bt_identifiability': 'WARNING_RESOLUTION_OR_BOUND' if warning else 'UNFLAGGED',
                    'disk_classification': bool(bt < .5), 'component_re_over_psf_proxy': ratios})
    return out


def _make_demo_config(config):
    out = Path(config['output_dir']).resolve()
    out.mkdir(parents=True, exist_ok=True)
    n = 121; scale = 0.03
    yy, xx = np.indices((n, n)); psf = np.exp(-((xx-(n-1)/2)**2+(yy-(n-1)/2)**2)/(2*1.2**2)); psf /= psf.sum()
    header = make_minimal_wcs_header(n, scale); header['BUNIT'] = 'Jy/pixel'
    header['CRVAL1'] = 150.; header['CRVAL2'] = 2.
    # The source and target are deliberately separate FITS products.
    source_model, _ = render_lenstronomy([{'Re_arcsec': .15, 'n': 1.3, 'q': .72, 'PA_deg': 20.}], (n,n), scale, psf, [2.4e-7])
    rng = np.random.default_rng(4); source_err = np.full((n,n), 2e-10); target_err = np.full((n,n), 3e-10)
    target_sci = rng.normal(0, target_err); source_sci = source_model + rng.normal(0, source_err)
    for name, image in [('source_sci.fits', source_sci), ('source_err.fits', source_err), ('target_sci.fits', target_sci), ('target_err.fits', target_err), ('source_psf.fits', psf), ('target_psf.fits', psf)]:
        fits.writeto(out / name, image, header if 'psf' not in name else make_minimal_wcs_header(n, scale), overwrite=True)
    demo = copy.deepcopy(config)
    demo.update({'demo_mode': True, 'ra': 150., 'dec': 2., 'target_injection_ra': 150., 'target_injection_dec': 2.,
                 'output_dir': str(out / 'JWST_redshifting_demo_output'), 'z_source': 1., 'z_target': 3.,
                 'photometry': {'F200W': (2.4e-7, 2e-8), 'F277W': (2.0e-7, 2e-8)}})
    for role in ('source','target'):
        demo[role].update({'sci_path': str(out / f'{role}_sci.fits'), 'err_path': str(out / f'{role}_err.fits'),
                            'seg_path': None, 'psf_path': str(out / f'{role}_psf.fits'), 'filter': 'F200W' if role=='source' else 'F444W'})
    return demo


def plot_psf(psf, output_dir):
    fig, ax = plt.subplots(figsize=(4,4)); im = ax.imshow(psf.array, origin='lower', cmap='viridis'); fig.colorbar(im, ax=ax)
    ax.set(title=f'{psf.role.title()} PSF (signed wings preserved)', xlabel='x [pix]', ylabel='y [pix]')
    path = Path(output_dir) / f'{psf.role}_psf_diagnostic.png'; fig.savefig(path, dpi=160, bbox_inches='tight'); plt.show(); plt.close(fig)
    return str(path)


def plot_noise_audit(audit, output_dir, role):
    fig, ax = plt.subplots(figsize=(5,3)); ax.hist(audit['samples'], bins=25, color='0.35')
    ax.axvline(audit['median_err'], color='tab:orange', label='median supplied ERR')
    ax.axvline(audit['empirical_rms'], color='tab:blue', label='empirical RMS')
    ax.legend(); ax.set(xlabel='patch RMS', ylabel='count', title=f'{role.title()} empirical-noise audit')
    path = Path(output_dir) / f'{role}_empirical_noise.png'; fig.savefig(path, dpi=160, bbox_inches='tight'); plt.show(); plt.close(fig)
    return str(path)


def display_preflight(checks):
    print('\n' + '='*68 + '\nJWST ARTIFICIAL REDSHIFTING PREFLIGHT\n' + '='*68)
    for check in checks: print(f'{check.name:30s} {check.state:7s} {check.detail}')
    ready = not any(c.state == 'FAIL' for c in checks)
    print('\n## READY TO RUN:', 'YES' if ready else 'NO')
    return ready


def exact_clean_recovery_self_test(psf, pixel_scale_arcsec, output_dir, pso_repeats):
    """Closed Lenstronomy-truth → Galight recovery test using the active target PSF."""
    npix = 81
    if npix % 2 == 0: npix -= 1
    truth = [{'role': 'single', 'Re_arcsec': 0.18, 'n': 1.6, 'q': 0.72, 'PA_deg': 20.}]
    image, _ = render_lenstronomy(truth, (npix, npix), pixel_scale_arcsec, psf.array, [1e-5])
    fit = run_galight_fit(image, np.full_like(image, 1e-10), None, psf.array, pixel_scale_arcsec,
                           truth, pso_repeats, Path(output_dir) / 'exact_clean_closure')
    row = fit['components'][0]
    recovery = {'truth_n': 1.6, 'truth_re_arcsec': .18, 'truth_q': .72,
                'fit_n': float(row['n_sersic']), 'fit_re_arcsec': float(row['R_sersic']),
                'fit_q': float(row['q']), 'chisq': float(fit['chisq']), 'bound_hits': fit['bound_hits']}
    recovery['delta_n'] = recovery['fit_n'] - recovery['truth_n']
    recovery['delta_re_fraction'] = recovery['fit_re_arcsec'] / recovery['truth_re_arcsec'] - 1
    recovery['delta_q'] = recovery['fit_q'] - recovery['truth_q']
    passed = (abs(recovery['delta_n']) < .25 and abs(recovery['delta_re_fraction']) < .10 and
              abs(recovery['delta_q']) < .10 and not recovery['bound_hits'])
    if not passed:
        raise PipelineFailure('TERMINAL_ERROR', 'CLEAN_RECOVERY_SELF_TEST_FAILURE',
                              f'Exact clean closure failed: {recovery}')
    atomic_write_json(Path(output_dir) / 'exact_clean_closure_receipt.json', recovery)
    return recovery


def run_pipeline(config):
    if config['demo_mode']: config = _make_demo_config(config)
    out = Path(config['output_dir']); out.mkdir(parents=True, exist_ok=True)
    append_log(out, 'single-object run started')
    atomic_write_json(out / 'frozen_config.json', config)
    checks = []
    try:
        source = load_dataset('source', config) if config['morphology_mode'] == 'FIT_NATIVE_FIRST' else None
        target = make_synthetic_target(config) if config['noise_mode'] == 'SYNTHETIC_BACKGROUND' else load_dataset('target', config)
        checks.extend([Check('SCI image', 'PASS', 'target SCI readable'), Check('WCS', 'PASS', 'source/target WCS valid'),
                       Check('Pixel scale', 'PASS', f'target {target.pixel_scale_arcsec:.5f} arcsec/pixel'), Check('Units', 'PASS', f'target {target.unit}')])
        source_psf = prepare_psf('source', config, source) if source else None
        target_psf = prepare_psf('target', config, target)
        checks.append(Check('PSF', 'PASS', 'source and target PSFs independently validated'))
        if source: plot_psf(source_psf, out)
        plot_psf(target_psf, out)
        closure = exact_clean_recovery_self_test(target_psf, target.pixel_scale_arcsec, out, config['pso_repeats'])
        checks.append(Check('Clean render recovery', 'PASS',
                            f"exact closure Δn={closure['delta_n']:.3g}, ΔRe/Re={closure['delta_re_fraction']:.3g}, Δq={closure['delta_q']:.3g}"))
        target_audit = empirical_noise_audit(target, config, protected_radius_pix=max(4, target_psf.array.shape[0]//2))
        checks.append(Check('ERR map', 'PASS', 'target ERR/RMS is finite and positive'))
        checks.append(Check('Empirical noise', target_audit['state'], f"empirical/ERR={target_audit['empirical_to_err']:.3f}"))
        if target_audit['segmentation_warning']: checks.append(Check('Segmentation', 'WARNING', target_audit['segmentation_warning']))
        else: checks.append(Check('Segmentation', 'PASS', 'external segmentation used for blank-sky masking'))
        plot_noise_audit(target_audit, out, 'target')
        if config['rescale_err']:
            target.err *= target_audit['empirical_to_err']; checks.append(Check('ERR rescaling', 'WARNING', 'explicitly enabled empirical scalar rescaling'))
        flux_plan = _photometry_plan(config)
        checks.append(Check('Wavelength support', 'PASS' if flux_plan['support'].startswith('PASS') else 'WARNING', flux_plan['support']))
        checks.append(Check('Cosmology', 'PASS', f"Fnu distance factor={flux_plan['cosmological_fnu_ratio']:.5g}"))
        if config['noise_mode'] == 'SYNTHETIC_BACKGROUND':
            checks.append(Check('Target location', 'WARNING', 'Synthetic background: no real-mosaic context/crowding diagnostic is available.'))
        elif config['target']['seg_path'] is None:
            checks.append(Check('Target location', 'WARNING', 'No segmentation at target location; verify the injection context is appropriate.'))
        else: checks.append(Check('Target location', 'PASS', 'target context segmentation supplied'))
        if config['morphology_mode'] == 'FIT_NATIVE_FIRST':
            start = _components_for_fit(config['structural_mode'], source.sci, source.pixel_scale_arcsec)
            native_input = run_galight_fit(source.sci, source.err, source.seg, source_psf.array, source.pixel_scale_arcsec, start, config['pso_repeats'], out / 'native_input_fit')
            truth_components = _as_model_components(native_input)
            native_clean, _ = render_lenstronomy(truth_components, source.sci.shape, source.pixel_scale_arcsec, source_psf.array)
            native_clean_fit = run_galight_fit(native_clean, source.err, None, source_psf.array, source.pixel_scale_arcsec, start, config['pso_repeats'], out / 'native_clean_fit')
            native_flux = float(native_clean.sum())
        else:
            supplied = config['parametric_input'].get('components', [])
            if not supplied:
                raise PipelineFailure('TERMINAL_ERROR', 'MISSING_PARAMETRIC_INPUT', 'PARAMETRIC mode requires PARAMETRIC_INPUT["components"].')
            truth_components = supplied
            base_shape = target.sci.shape
            source_psf = target_psf; native_fluxes = [float(x['flux_jy']) for x in supplied]
            native_clean, _ = render_lenstronomy(truth_components, base_shape, target.pixel_scale_arcsec, target_psf.array, native_fluxes)
            native_clean_fit = run_galight_fit(native_clean, target.err, None, target_psf.array, target.pixel_scale_arcsec,
                                                _components_for_fit(config['structural_mode'], native_clean, target.pixel_scale_arcsec), config['pso_repeats'], out / 'native_clean_fit')
            native_input = {'components': [], 'chisq': np.nan, 'bound_hits': [], 'source_result': []}; native_flux = float(native_clean.sum())
        checks.append(Check('Native clean recovery', 'PASS', 'Lenstronomy clean model recovered with Galight'))
        angular = Planck18.angular_diameter_distance(config['z_source']).value / Planck18.angular_diameter_distance(config['z_target']).value
        target_components = copy.deepcopy(truth_components)
        for c in target_components:
            c['R_sersic'] = float(c.get('R_sersic', c.get('Re_arcsec'))) * angular
            c['center_x'] = float(c.get('center_x', 0.)) * angular; c['center_y'] = float(c.get('center_y', 0.)) * angular
        if flux_plan['target_fnu_jy'] is not None:
            target_flux = float(flux_plan['target_fnu_jy'])
        else:
            target_flux = native_flux * flux_plan['cosmological_fnu_ratio'] * config['luminosity_evolution_factor']
        target_clean, _ = render_lenstronomy(target_components, target.sci.shape, target.pixel_scale_arcsec, target_psf.array)
        if target_clean.sum() <= 0: raise PipelineFailure('TERMINAL_ERROR', 'INVALID_TARGET_MODEL', 'Target clean model has nonpositive flux.')
        target_clean *= target_flux / target_clean.sum()
        flux_error = abs(target_clean.sum() / target_flux - 1)
        checks.append(Check('Flux conservation', 'PASS' if flux_error < 1e-10 else 'FAIL', f'fractional error={flux_error:.2e}'))
        if flux_error >= 1e-10: raise PipelineFailure('TERMINAL_ERROR', 'FLUX_CONSERVATION_FAILURE', 'Clean target model does not conserve requested flux.')
        clean_fit = run_galight_fit(target_clean, target.err, None, target_psf.array, target.pixel_scale_arcsec,
                                    _components_for_fit(config['structural_mode'], target_clean, target.pixel_scale_arcsec), config['pso_repeats'], out / 'target_clean_fit')
        injected = inject_real_background(target, target_clean, config)
        checks.append(Check('Background treatment', 'PASS', injected['noise_statement']))
        real_fit = run_galight_fit(injected['image'], injected['err'], target.seg, target_psf.array, target.pixel_scale_arcsec,
                                   _components_for_fit(config['structural_mode'], injected['image'], target.pixel_scale_arcsec), config['pso_repeats'], out / 'target_real_fit')
        checks.append(Check('Target clean recovery', 'PASS', 'Galight target-clean recovery completed'))
        checks.append(Check('Target real recovery', 'PASS', 'Galight target-real recovery completed'))
        result = {'status': 'OK', 'timestamp_utc': _now(), 'config': config, 'checks': [asdict(c) for c in checks],
                  'flux_plan': flux_plan, 'angular_size_ratio': float(angular), 'target_noise_audit': target_audit,
                  'exact_clean_closure': closure,
                  'native_input': _quality_summary(native_input, source_psf, config) if source else native_input,
                  'native_clean': _quality_summary(native_clean_fit, source_psf, config),
                  'target_clean': _quality_summary(clean_fit, target_psf, config),
                  'target_real': _quality_summary(real_fit, target_psf, config),
                  'flux_requested_jy': target_flux, 'flux_rendered_jy': float(target_clean.sum()),
                  'background_noise_draws': injected['background_noise_draws'], 'source_poisson_added': injected['source_poisson_added'],
                  'noise_statement': injected['noise_statement']}
        display_preflight([Check(**x) for x in result['checks']])
        save_results(result, images={'native_clean': native_clean, 'target_clean': target_clean, 'target_real': injected['image'], 'target_residual': real_fit['residual']})
        append_log(out, 'single-object run completed status=OK')
        return result
    except PipelineFailure as exc:
        checks.append(Check('Pipeline', 'FAIL', exc.message)); display_preflight(checks)
        result = {'status': exc.status, 'error_code': exc.code, 'error_message': exc.message, 'checks': [asdict(c) for c in checks], 'timestamp_utc': _now(), 'config': config}
        atomic_write_json(out / 'result.json', result); append_log(out, f'run failed {exc.status}:{exc.code} {exc.message}')
        raise
    except Exception as exc:
        result = {'status': 'RETRYABLE_ERROR', 'error_code': 'UNCAUGHT_EXCEPTION', 'error_message': str(exc), 'traceback': traceback.format_exc(), 'timestamp_utc': _now(), 'config': config}
        atomic_write_json(out / 'result.json', result); append_log(out, f'retryable exception {exc}')
        raise PipelineFailure('RETRYABLE_ERROR', 'UNCAUGHT_EXCEPTION', str(exc)) from exc


def show_results(result):
    print('\nRESULT STATUS:', result['status'])
    if result['status'] != 'OK': print(result.get('error_code'), result.get('error_message')); return
    for stage in ('native_clean', 'target_clean', 'target_real'):
        q = result[stage]; print(f"{stage:14s} chi2={q['chisq']:.4g} bounds={q['bound_hits']} n={q.get('single_n', np.nan):.4g} B/T={q.get('bt', np.nan):.4g}")
    print('Interpret target-clean → target-real as observational/context degradation, not intrinsic evolution.')


def save_results(result, images=None):
    out = Path(result['config']['output_dir']); out.mkdir(parents=True, exist_ok=True)
    atomic_write_json(out / 'result.json', result)
    atomic_write_json(out / 'provenance.json', {'timestamp_utc': _now(), 'python': sys.version, 'platform': platform.platform(),
                                                 'numpy': np.__version__, 'astropy': __import__('astropy').__version__,
                                                 'lenstronomy': __import__('lenstronomy').__version__, 'galight': getattr(__import__('galight'), '__version__', 'unknown')})
    if images:
        fig, axes = plt.subplots(1, len(images), figsize=(4*len(images), 4))
        axes = np.atleast_1d(axes)
        for ax, (name, image) in zip(axes, images.items()):
            im = ax.imshow(image, origin='lower', cmap='magma'); ax.set_title(name); fig.colorbar(im, ax=ax, fraction=.046)
        fig.savefig(out / 'stage_images.png', dpi=160, bbox_inches='tight'); plt.show(); plt.close(fig)
    return out / 'result.json'


def case_should_run(existing, force_ids):
    if existing is None: return True
    return existing.get('status') == 'RETRYABLE_ERROR' or str(existing.get('case_id')) in {str(x) for x in force_ids}


def load_valid_checkpoints(checkpoint_dir):
    rows = {}
    for path in Path(checkpoint_dir).glob('case_*.json'):
        try:
            row = json.loads(path.read_text())
            cid = str(row['case_id'])
            if cid in rows: raise PipelineFailure('TERMINAL_ERROR', 'DUPLICATE_CHECKPOINT', f'Duplicate checkpoint for case {cid}.')
            rows[cid] = row
        except PipelineFailure: raise
        except Exception: continue  # incomplete/non-JSON checkpoint is retryable by absence
    return rows


def archive_prior_receipt(checkpoint_dir, case_id, receipt):
    """Keep a prior ERROR/forced-run receipt without duplicating the active case row."""
    stamp = dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    path = Path(checkpoint_dir) / 'history' / f'case_{case_id}_{stamp}.json'
    atomic_write_json(path, receipt)
    return path


def run_batch(config):
    if not config['batch_catalog_path']:
        raise PipelineFailure('TERMINAL_ERROR', 'MISSING_BATCH_CATALOG', 'Set BATCH_CATALOG_PATH for RUN_MODE=BATCH.')
    out = Path(config['output_dir']); ckpt = out / 'checkpoints'; ckpt.mkdir(parents=True, exist_ok=True)
    with open(config['batch_catalog_path'], newline='', encoding='utf-8') as handle: rows = list(csv.DictReader(handle))
    existing = load_valid_checkpoints(ckpt); merged = []
    for row in rows:
        cid = str(row[config['batch_column_map']['case_id']]); old = existing.get(cid)
        if not case_should_run(old, config['force_rerun_ids']): merged.append(old); continue
        case_cfg = copy.deepcopy(config); case_cfg['run_mode'] = 'SINGLE'; case_cfg['ra'] = float(row[config['batch_column_map']['ra']]); case_cfg['dec'] = float(row[config['batch_column_map']['dec']]); case_cfg['z_source'] = float(row[config['batch_column_map']['z_source']]); case_cfg['output_dir'] = str(out / 'cases' / cid)
        try:
            res = run_pipeline(case_cfg); receipt = {'case_id': cid, 'status': res['status'], 'result_path': str(Path(case_cfg['output_dir']) / 'result.json'), 'timestamp_utc': _now()}
        except PipelineFailure as exc:
            receipt = {'case_id': cid, 'status': exc.status, 'error_code': exc.code, 'error_message': exc.message, 'timestamp_utc': _now()}
        if old is not None:
            receipt['supersedes_status'] = old.get('status')
            receipt['prior_receipt_path'] = str(archive_prior_receipt(ckpt, cid, old))
        atomic_write_json(ckpt / f'case_{cid}.json', receipt); merged.append(receipt); append_log(out, f'case={cid} status={receipt["status"]}')
    ids = [str(x['case_id']) for x in merged]
    if len(ids) != len(set(ids)): raise PipelineFailure('TERMINAL_ERROR', 'DUPLICATE_BATCH_ROWS', 'Batch summary would contain duplicate case IDs.')
    atomic_write_json(out / 'batch_summary.json', {'rows': merged, 'timestamp_utc': _now()})
    return merged


def run_internal_smoke_tests():
    cfg = normalize_config({'DEMO_MODE': True, 'RUN_MODE': 'SINGLE', 'SOURCE_REDSHIFT': 1., 'TARGET_REDSHIFT': 3.})
    cfg['source']['err_path'] = 'source_err.fits'; cfg['target']['err_path'] = 'target_err.fits'
    assert cfg['source']['err_path'] != cfg['target']['err_path']
    assert case_should_run({'status': 'OK', 'case_id': '1'}, set()) is False
    assert case_should_run({'status': 'TERMINAL_ERROR', 'case_id': '1'}, set()) is False
    assert case_should_run({'status': 'RETRYABLE_ERROR', 'case_id': '1'}, set()) is True
    assert case_should_run({'status': 'OK', 'case_id': '1'}, {'1'}) is True
    return 'Internal helper smoke tests: PASS'


## 6–14. Inspect, preflight, render, inject, recover, and diagnose

In [ ]:
# One-cell execution. In normal use, edit only USER SETTINGS above and Run All.
CONFIG = normalize_config(globals())
print(run_internal_smoke_tests())
if CONFIG['run_mode'] == 'SINGLE':
    result = run_pipeline(CONFIG)
    show_results(result)
else:
    result = run_batch(CONFIG)
    print(f'Batch receipts written: {len(result)}')


## 15–16. Results, saved outputs, and provenance

`result.json` is the machine-readable single-object receipt. `frozen_config.json`,
`provenance.json`, `run.log`, PSF/noise figures, and stage images are stored in
`OUTPUT_DIR`. Batch mode additionally writes one atomic JSON checkpoint per
case and a duplicate-checked `batch_summary.json`.

## 17. Optional batch/catalog mode

Set `RUN_MODE = 'BATCH'`, `BATCH_CATALOG_PATH`, and `BATCH_COLUMN_MAP` in the
user settings cell. Resume skips `OK`, `WARNING`, and `TERMINAL_ERROR` receipts,
retries `RETRYABLE_ERROR`/incomplete cases, and permits selected reruns through
`FORCE_RERUN_IDS`. Do not use a force rerun to erase scientific evidence.

## 18. Limitations and interpretation

- `native-clean → target-clean` measures resolution/redshift effects; `target-clean → target-real` measures context/observational degradation.
- A final drizzled mosaic cannot in general reconstruct exact detector-level
  source-Poisson covariance. The default deterministic mode is intentionally
  honest about this; do not create Poisson noise from MJy/sr alone.
- The empirical blank-sky diagnostic measures correlated-noise mismatch but
  does not silently rescale errors.
- Resolution and B/T flags are warnings/receipts, not a universal morphology
  threshold. Interpret compact multi-component decompositions cautiously.
- This notebook masks segmented neighbours in its simple public fit. Crowded
  fields may require explicit jointly modeled neighbours and field-specific
  validation before physical interpretation.
